# Hardline audio — OmniVoice (voice.json driven)

Cell 2 asks for two uploads: first your voice sample, then the voice.json file(s). Output `hardline_audio.zip` has `<id>/<sceneId>.wav`.

**Runtime → GPU (T4)**, run top to bottom. Unzip so each `<id>/` folder lands inside `hardline-video/public/audio/`, then `npm run make:all`.

In [ ]:
# CELL 1 — install + load model (proven recipe; no restart)
!pip install -q omnivoice
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
import torch, torchaudio
from omnivoice import OmniVoice
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print('Device:', device)
print('Loading OmniVoice (first run downloads ~3-5 GB)...')
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('Model loaded!')

In [ ]:
# CELL 2 — two separate uploads
from google.colab import files
AUDIO_EXT = ('.mp3', '.wav', '.m4a', '.flac', '.ogg', '.aac')

print('STEP 1 of 2 — upload your VOICE SAMPLE (.mp3 / .wav):')
up_audio = files.upload()
REF_AUDIO = next((n for n in up_audio if n.lower().endswith(AUDIO_EXT)), None)

print('\nSTEP 2 of 2 — upload your voice.json file(s):')
up_json = files.upload()
VOICE_JSONS = sorted(n for n in up_json if n.lower().endswith('.json'))

print('\nvoice sample:', REF_AUDIO if REF_AUDIO else '!! MISSING — re-run and pick an audio file in STEP 1')
print('voice jsons :', VOICE_JSONS if VOICE_JSONS else '!! MISSING — re-run and pick .json files in STEP 2')

In [ ]:
# CELL 3 — settings. Leave REF_TEXT '' to auto-transcribe the sample.
REF_TEXT  = ''
NUM_STEPS = 32   # 16 = quick draft, 32 = final quality
SPEED     = 1.0

In [ ]:
# CELL 4 — generate audio for every scene of every voice.json
import os, json, subprocess
assert REF_AUDIO, 'Upload a voice sample in Cell 2 (STEP 1) first.'
assert VOICE_JSONS, 'Upload at least one voice.json in Cell 2 (STEP 2) first.'

REF_WAV = 'ref.wav'
subprocess.run(['ffmpeg','-y','-i',REF_AUDIO,'-ar','24000','-ac','1',REF_WAV],
               check=True, capture_output=True)

def synth(text, path):
    kwargs = dict(text=text, ref_audio=REF_WAV, num_step=int(NUM_STEPS), speed=float(SPEED))
    if REF_TEXT.strip():
        kwargs['ref_text'] = REF_TEXT
    audio = model.generate(**kwargs)
    t = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
    if t.dim() == 1:
        t = t.unsqueeze(0)
    torchaudio.save(path, t.cpu().float(), 24000)
    return t.shape[-1] / 24000.0

for vj in VOICE_JSONS:
    spec = json.load(open(vj, encoding='utf-8'))
    vid = spec['id']
    title = spec.get('title', vid)
    os.makedirs(f'out/{vid}', exist_ok=True)
    print(f'== {vid} :: {title} ==')
    durations = {}
    for scene in spec['scenes']:
        durations[scene['id']] = round(synth(scene['text'], f"out/{vid}/{scene['id']}.wav"), 3)
        print('   ', scene['id'], durations[scene['id']], 's')
    json.dump(durations, open(f'out/{vid}/durations.json','w'), indent=2)
print('done')

In [ ]:
# CELL 5 — zip + download (contains <id>/<sceneId>.wav for every video)
import shutil
shutil.make_archive('hardline_audio', 'zip', 'out')
from google.colab import files
files.download('hardline_audio.zip')